# AI Auto-Scaling Load Balancer (Đồ Án Level 7)

Notebook này sử dụng **Reinforcement Learning (PPO)** để tự động điều chỉnh tải (Auto-Scaling) cho 3 máy chủ Web. Nó sẽ học cách:
- Tắt máy vắng khách ban đêm (Tiết kiệm điện)
- Bật máy mạnh mẽ ban ngày
- Bật dự phòng S3 lúc khẩn cấp (Chống sập web)

**1. Cài đặt thư viện**

In [ ]:
!pip install gymnasium stable-baselines3 shimmy

**2. Import Môi trường Mô phỏng (SimEnv)**
Chúng ta copy code của file `lb_env_sim.py` vào đây.

In [ ]:
%%writefile lb_env_sim.py
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import math

class HAProxySimEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, max_steps=200):
        super().__init__()
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(12,), dtype=np.float32)
        self.WEIGHT_ACTIONS = [
            (10, 0, 0), (0, 10, 0), (5, 5, 0), (8, 2, 0), (2, 8, 0), (4, 4, 2), (3, 3, 4)
        ]
        self.action_space = spaces.Discrete(len(self.WEIGHT_ACTIONS))
        self.max_steps = max_steps
        self.reset()
        
    def _simulate_traffic(self, time_step):
        base_traffic = 50 + 40 * math.sin(time_step * (2 * math.pi / 60))
        if time_step % 45 > 40: return base_traffic + 100
        return max(10, base_traffic)

    def _math_model(self, incoming_reqs, weight):
        capacity = 100
        handled_reqs = incoming_reqs * (weight / 10.0)
        if weight == 0: return 0.0, 10.0, 0
        cpu = min(100.0, (handled_reqs / capacity) * 100)
        if cpu < 50: lat = 20 + cpu
        elif cpu < 80: lat = 20 + cpu * 1.5
        elif cpu < 95: lat = 100 + (cpu - 80) * 20
        else: lat = 800 + (cpu - 95) * 100
        return cpu, min(2000.0, lat), min(50, handled_reqs)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.step_count = 0
        self.last_action = 0
        return self._get_obs(10, 0, 0), {}

    def _get_obs(self, w1, w2, w3):
        traffic = self._simulate_traffic(self.step_count)
        c1, l1, _ = self._math_model(traffic, w1)
        c2, l2, _ = self._math_model(traffic, w2)
        c3, l3, _ = self._math_model(traffic, w3)
        self.current_state = {"c1":c1, "l1":l1, "w1":w1, "c2":c2, "l2":l2, "w2":w2, "c3":c3, "l3":l3, "w3":w3}
        return np.array([c1/100., 0.5, 0.5, l1/2000., c2/100., 0.5, 0.5, l2/2000., c3/100., 0.5, 0.5, l3/2000.], dtype=np.float32)

    def step(self, action):
        w1, w2, w3 = self.WEIGHT_ACTIONS[action]
        obs = self._get_obs(w1, w2, w3)
        stats = [{"cpu": self.current_state["c1"], "lat": self.current_state["l1"], "w": w1},
                 {"cpu": self.current_state["c2"], "lat": self.current_state["l2"], "w": w2},
                 {"cpu": self.current_state["c3"], "lat": self.current_state["l3"], "w": w3}]
        active_srvs = [s for s in stats if s["w"] > 0]
        if not active_srvs: return obs, -200, True, False, {}
        max_lat, max_cpu = max([s["lat"] for s in active_srvs]), max([s["cpu"] for s in active_srvs])
        reward = 0.0
        done = True if max_lat > 2000 else False
        if max_lat < 100: reward += 15
        elif max_lat < 300: reward += 5
        elif max_lat < 800: reward -= 5
        else: reward -= 20
        if w1 > 0: reward -= 2
        if w2 > 0: reward -= 2
        if w3 > 0: reward -= 5
        if max_cpu > 85: reward -= 60  # Phạt rát hơn để tránh Sập CPU
        elif max_cpu > 70: reward -= 15
        elif max_cpu < 50: reward += 5
        if len(active_srvs) >= 2:
            if (max_cpu - min([s["cpu"] for s in active_srvs])) > 30: reward -= 3
        if action != self.last_action: reward -= 0.5  # Giảm phạt vì tội thay đổi đi (khiến nó dạn dĩ hơn)
        self.last_action = action
        self.step_count += 1
        info = {"traffic": self._simulate_traffic(self.step_count), "reward": reward, 
                "cpu_s1": self.current_state["c1"], "cpu_s2": self.current_state["c2"], "cpu_s3": self.current_state["c3"],
                "w1": w1, "w2": w2, "w3": w3}
        return obs, reward, done, self.step_count >= self.max_steps, info

**3. Huấn luyện Model bằng PPO**
Sử dụng Entropy Coefficient (ent_coef) để ép AI Tò mò Khám phá nhiều hơn.

In [ ]:
from stable_baselines3 import PPO
from lb_env_sim import HAProxySimEnv
import time

env = HAProxySimEnv(max_steps=500)
# ent_coef = 0.05 ép AI năng động khám phá
model = PPO("MlpPolicy", env, verbose=1, learning_rate=5e-4, n_steps=2048, batch_size=64, ent_coef=0.05)

print("Bắt đầu huấn luyện...")
start = time.time()
model.learn(total_timesteps=150000)
print(f"Huấn luyện xong trong {time.time()-start:.1f}s")

model.save("ha_rl_trained_model")
print("Đã lưu model: ha_rl_trained_model.zip")

**4. Biểu diễn Trực quan (Visualization)**
Vẽ đồ thị xem AI đã học được cách tắt máy ban đêm và gọi diện backup ban ngày như thế nào.

In [ ]:
import matplotlib.pyplot as plt

env = HAProxySimEnv(max_steps=200)
obs, _ = env.reset()

history = {"traffic": [], "cpu1": [], "cpu2": [], "cpu3": [], "w1": [], "w2": [], "w3": []}

for _ in range(200):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, tr, info = env.step(action)
    
    history["traffic"].append(info["traffic"])
    history["cpu1"].append(info["cpu_s1"])
    history["cpu2"].append(info["cpu_s2"])
    history["cpu3"].append(info["cpu_s3"])
    history["w1"].append(info["w1"])
    history["w2"].append(info["w2"])
    history["w3"].append(info["w3"])
    if done or tr: break

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10))

ax1.plot(history["traffic"], color='purple', label="Web Traffic (Requests)")
ax1.set_title("Lượng Traffic Mô Phỏng (Đỉnh nhọn là Sale)")
ax1.legend()

ax2.plot(history["w1"], label="Weight Server 1", linestyle='--')
ax2.plot(history["w2"], label="Weight Server 2", linestyle='--')
ax2.plot(history["w3"], label="Weight Server 3 (Backup)", linestyle='--')
ax2.set_title("Trọng số do AI Điều Khiển (0 = Server ngủ đông)")
ax2.legend()

ax3.plot(history["cpu1"], label="CPU Server 1 (%)")
ax3.plot(history["cpu2"], label="CPU Server 2 (%)")
ax3.plot(history["cpu3"], label="CPU Server 3 (%)")
ax3.axhline(85, color='red', linestyle=':', label="Ngưỡng Nguy Hiểm (85%)")
ax3.set_title("CPU Máy Chủ (AI giữ tất cả dưới vạch đỏ)")
ax3.legend()

plt.tight_layout()
plt.show()